# Headless Dafne Thigh segmentation

Runs the local Dafne Thigh model on all FATFRACTION NIfTI stacks found under `myosegmenTUM/`.  
No GUI required — uses `dafne_dl.DynamicDLModel` directly.  

**Kernel:** `dafne_clean`

In [1]:
import glob
import os
import numpy as np
import SimpleITK as sitk
from dafne_dl import DynamicDLModel

C:\Users\docto\miniconda3\envs\dafne_clean\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# --- paths ---
MODEL_PATH  = r"C:\Users\docto\AppData\Local\Dafne-imaging\Dafne\models\Thigh_1774532147.model"
IMAGE_GLOB  = "myosegmenTUM/*/ImageData/*FATFRACTION/*FATFRACTION_stack*.nii"
OUTPUT_DIR  = "dafne_thigh_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# load model once — DynamicDLModel.Load reads the local .model file
model = DynamicDLModel.Load(open(MODEL_PATH, "rb"))
print("Model loaded:", MODEL_PATH)


Model loaded: C:\Users\docto\AppData\Local\Dafne-imaging\Dafne\models\Thigh_1774532147.model


In [4]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f"Found {len(image_files)} images:")
for p in image_files:
    print(" ", p)

Found 54 images:
  myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack1.nii
  myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack2.nii
  myosegmenTUM\HV001_2\ImageData\HV001_2_FATFRACTION\HV001_2_FATFRACTION_stack1.nii
  myosegmenTUM\HV001_2\ImageData\HV001_2_FATFRACTION\HV001_2_FATFRACTION_stack2.nii
  myosegmenTUM\HV001_3\ImageData\HV001_3_FATFRACTION\HV001_3_FATFRACTION_stack1.nii
  myosegmenTUM\HV001_3\ImageData\HV001_3_FATFRACTION\HV001_3_FATFRACTION_stack2.nii
  myosegmenTUM\HV002_1\ImageData\HV002_1_FATFRACTION\HV002_1_FATFRACTION_stack1.nii
  myosegmenTUM\HV002_1\ImageData\HV002_1_FATFRACTION\HV002_1_FATFRACTION_stack2.nii
  myosegmenTUM\HV002_2\ImageData\HV002_2_FATFRACTION\HV002_2_FATFRACTION_stack1.nii
  myosegmenTUM\HV002_2\ImageData\HV002_2_FATFRACTION\HV002_2_FATFRACTION_stack2.nii
  myosegmenTUM\HV002_3\ImageData\HV002_3_FATFRACTION\HV002_3_FATFRACTION_stack1.nii
  myosegmenTUM\HV002_3\ImageData\HV002_3_FATFRACTION\HV002_

In [ ]:
# run segmentation slice-by-slice and save one .npz per stack
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_dafne_thigh.npz")

    if os.path.exists(out_path):
        print(f"Skipping (already done): {out_path}")
        continue

    print(f"\nProcessing: {nii_path}")

    img_sitk   = sitk.ReadImage(nii_path)
    img_array  = sitk.GetArrayFromImage(img_sitk).astype(float)  # (slices, H, W)
    spacing    = img_sitk.GetSpacing()                            # (x_mm, y_mm, z_mm)
    resolution = [spacing[0], spacing[1]]                         # 2D in-plane spacing

    print(f"  Shape: {img_array.shape}  Resolution: {resolution}")

    all_masks = {}  # {muscle_name: 3D uint8 array (slices, H, W)}

    for slice_idx in range(img_array.shape[0]):
        slice_2d = img_array[slice_idx]
        out = model({
            "image": slice_2d,
            "resolution": resolution,
            "split_laterality": True,
            "classification": "Thigh",
        })

        for muscle_name, mask in out.items():
            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = np.asarray(mask, dtype=np.uint8)

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f"  slice {slice_idx + 1}/{img_array.shape[0]} done")

    np.savez_compressed(out_path, **all_masks)
    print(f"  Saved → {out_path}")
    print(f"  Muscles: {list(all_masks.keys())}")

print("\nAll done.")

In [7]:
# quick sanity check — reload one result and show muscle names + voxel counts
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.npz")))
if results:
    sample = np.load(results[0])
    print("Sample file:", results[0])
    for name in sample.files:
        arr = sample[name]
        print(f"  {name}: shape={arr.shape}  positive voxels={arr.sum()}")

Sample file: dafne_thigh_results\HV001_1_FATFRACTION_stack1_dafne_thigh.npz
  Vastus Lateralis_R: shape=(65, 672, 672)  positive voxels=11707
  Vastus Lateralis_L: shape=(65, 672, 672)  positive voxels=6258
  Vastus Medialis_R: shape=(65, 672, 672)  positive voxels=72799
  Vastus Medialis_L: shape=(65, 672, 672)  positive voxels=70797
  Vastus Intermedius_R: shape=(65, 672, 672)  positive voxels=47175
  Vastus Intermedius_L: shape=(65, 672, 672)  positive voxels=56194
  Rectus Femoris_R: shape=(65, 672, 672)  positive voxels=4891
  Rectus Femoris_L: shape=(65, 672, 672)  positive voxels=2012
  Sartorius_R: shape=(65, 672, 672)  positive voxels=11381
  Sartorius_L: shape=(65, 672, 672)  positive voxels=23461
  Gracilis_R: shape=(65, 672, 672)  positive voxels=175018
  Gracilis_L: shape=(65, 672, 672)  positive voxels=171869
  Adductor Magnus_R: shape=(65, 672, 672)  positive voxels=151138
  Adductor Magnus_L: shape=(65, 672, 672)  positive voxels=133288
  Semimembranosus_R: shape=(65, 6